<a href="https://colab.research.google.com/github/anjalikaushik20/Problem-solving/blob/master/templates/23_cross_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
from IPython.utils.sysinfo import num_cpus
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
      head_dim = self.d_model // self.num_heads
      Q = self.W_q(x_q)
      K = self.W_k(x_kv)
      V = self.W_v(x_kv)

      B, Tq, D = Q.shape
      B, Tk, D = K.shape
      B, Tv, D = V.shape

      Q = Q.reshape(B, Tq, self.num_heads, head_dim).transpose(1, 2)
      K = K.reshape(B, Tk, self.num_heads, head_dim).transpose(1, 2)
      V = V.reshape(B, Tv, self.num_heads, head_dim).transpose(1, 2)

      O_1 = (Q@K.transpose(-2, -1))/math.sqrt(head_dim)
      O_2 = torch.softmax(O_1, dim=-1)
      O = O_2 @ V

      O = self.W_o(O.transpose(1, 2).reshape(B, Tq, D))
      return O

        # pass  # Q from x_q, K/V from x_kv, no causal mask

In [4]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (4.4ms)
  ✅ [2/4] Q and KV different lengths (3.7ms)
  ✅ [3/4] No causal mask — all KV affects all Q (50.7ms)
  ✅ [4/4] Gradient flow (36.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (95.0ms total)
  Progress saved. Run status() to see your dashboard.

